## A. Model Training

### ✅ 1. Setup + Load Data

In [10]:
import pandas as pd
import numpy as np

# Load dataset
train_df = pd.read_parquet("../../data/merged_data/train.parquet")
val_df = pd.read_parquet("../../data/merged_data/val.parquet")


### ✅ 2. Print top 5 rows in training and validation datasets.

In [7]:
# Inspect training set
print(train_df.shape)
train_df.head()

(157454, 26)


,job_title,experience_years,education_level,skills_count,industry,company_size,country,remote_work,certifications,cost_of_living_index,...,salary_per_year_experience,salary_vs_country_mean,rent_burden,grocery_burden,developing_country_flag,col_adjusted_salary,ppp_index,experience_bracket,salary_rent_ratio,salary
0,0.100075,-1.321133,-1.414437,0.365232,0.364795,-0.001780,-0.317017,-1.220253,-0.288613,1.038775,...,0.496078,-1.012315,3.118945,1.844508,-0.352775,-1.261591,-0.352690,-1.692395,-1.015439,107998
1,1.849836,-0.991138,1.414462,-1.642961,-1.497897,-1.419139,0.939576,1.229587,0.882982,0.244958,...,0.443648,0.183994,-0.277892,-0.219669,-0.352775,-0.001115,-0.821176,-0.952119,-0.208187,173088
2,0.100075,-0.661142,0.707237,-0.547583,-1.497897,-0.710460,-2.182841,-1.220253,1.468779,-2.663479,...,-0.376680,-0.216379,-1.305927,-1.326272,2.834669,1.993588,-2.156993,-0.211843,2.261861,90991
3,0.415820,-0.991138,1.414462,-0.912709,1.104216,-1.419139,-0.317017,1.229587,-0.874411,1.038775,...,0.124144,-0.269810,2.257110,0.948870,-0.352775,-1.059811,-0.352690,-0.952119,-0.938864,130987
4,-0.255860,-0.991138,-0.707212,0.182669,-1.213615,-0.001780,0.611832,-1.220253,0.882982,0.058178,...,-0.028919,-1.590920,0.423143,0.534651,-0.352775,-0.720558,-0.093124,-0.952119,-0.577571,110818


In [2]:
# Inspect validation set
print(val_df.shape)
val_df.head()

(33740, 26)


,job_title,experience_years,education_level,skills_count,industry,company_size,country,remote_work,certifications,cost_of_living_index,...,salary_per_year_experience,salary_vs_country_mean,rent_burden,grocery_burden,developing_country_flag,col_adjusted_salary,ppp_index,experience_bracket,salary_rent_ratio,salary
0,-1.539964,-0.166148,0.707237,-0.730146,-1.497897,0.706899,0.611832,1.229587,-0.874411,0.058178,...,-0.297546,-0.298214,-0.164518,-0.471857,-0.352775,-0.182794,-0.093124,-0.211843,-0.289501,150842
1,-1.539964,0.493844,-0.707212,-1.460398,0.896274,1.415579,1.581300,1.229587,-1.460209,0.618519,...,-0.423439,-0.548628,0.206127,0.104945,-0.352775,-0.367688,1.622547,0.528433,-0.490927,164730
2,0.282537,-0.661142,-0.707212,-1.642961,-0.466680,1.415579,-0.317017,-1.220253,-0.288613,1.038775,...,-0.169376,-0.326429,2.312030,1.005943,-0.352775,-1.075198,-0.352690,-0.211843,-0.944703,129234
3,-0.871778,0.988837,-0.707212,1.095485,1.538154,-0.001780,1.581300,-1.220253,1.468779,0.618519,...,-0.472398,-0.116962,0.056207,-0.147475,-0.352775,-0.218256,1.622547,1.268710,-0.419186,178095
4,0.100075,-0.991138,0.707237,1.095485,0.067771,0.706899,-0.317017,1.229587,-0.288613,1.038775,...,0.198766,0.047779,1.974400,0.655071,-0.352775,-0.973505,-0.352690,-0.952119,-0.906110,140820


### ✅ 3. Train Models using Linear Regression and XGBoost.

In [8]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


# Separate features and target
X_train = train_df.drop(columns=['salary'])
y_train = train_df['salary']

X_val = val_df.drop(columns=['salary'])
y_val = val_df['salary']


# Initialize and train
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict and Evaluate
lr_preds = lr_model.predict(X_val)
print(f"Linear Regression RMSE: {np.sqrt(mean_squared_error(y_val, lr_preds)):.2f}")
print(f"Linear Regression R2: {r2_score(y_val, lr_preds):.2f}")



# Define the model
xgb_regressor = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# Setup Grid Search
grid_search = GridSearchCV(
    estimator=xgb_regressor,
    param_grid=param_grid,
    cv=3, 
    scoring='neg_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

# Fit Grid Search
grid_search.fit(X_train, y_train)

# Best results
print(f"Best Parameters: {grid_search.best_params_}")
best_xgb = grid_search.best_estimator_

Linear Regression RMSE: 0.00
Linear Regression R2: 1.00
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best Parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}


In [9]:
# Final evaluation
final_preds = best_xgb.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, final_preds))
r2 = r2_score(y_val, final_preds)

print(f"Tuned XGBoost RMSE: {rmse:.2f}")
print(f"Tuned XGBoost R2 Score: {r2:.2f}")

Tuned XGBoost RMSE: 284.45
Tuned XGBoost R2 Score: 1.00
